## DATA PREPROCESSING TANPA STEMMING

- word lengthening normalization (melakukan normalisasi kata pada tweet agar tidak terdapat pengulangan huruf secara berlebihan.
- slang normalization (mengubah kata-kata slang di indonesia termasuk singkatan, yang mana ini menggunakan kamus yang telah dibuat oleh orang lain, ada 15.396 kata pada kamusnya)
- tokenization  (membuat token-token berdasarkan kata)

In [1]:
# 1. Import library utama
import pandas as pd
import re
import os

In [2]:
# 2. Load dataset hasil data cleaning
df = pd.read_csv('../datacleaningcopy/data_cleaning_final.csv')

print("Dataset berhasil dimuat.")
print("Jumlah data:", len(df))
df.head()

Dataset berhasil dimuat.
Jumlah data: 13192


,no,timestamp,teks
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...


In [4]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

df = pd.read_csv('../datapreprocessing-no-stem/data_preprocessing_final_no_stem.csv')
df.head()

,no,timestamp,teks,teks_processed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara Ingat yg ini!!!,ADIL loh untuk yang punya kebijakan publik negara ingat yang ini ! !
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan Sporadis Apalagi Selektif Hanya kepada Media yang Berseberangan,tertibkan media online DPR pemerintah jangan sporadis apalagi selektif hanya kepada media yang berseberangan
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa trutama utk negara tiongkok pak!! bahaya tersembunyi martabat RI,harus dievaluasi lagi kebijakan bebas visa terutama untuk negara tiongkok pak ! ! bahaya tersembunyi martabat RI
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin UU Namun kebebasan tersebut tdk harus kebablasan sehingga menabrak aturan aturan logis,kebebasan bersuara berpendapat memang dijamin UU tetapi kebebasan tersebut tidak harus kebablasan sehingga menabrak aturan aturan logis


## WORD LENGTHENING NORMALIZATION

In [ ]:
# 1.1 Fungsi untuk normalisasi word lengthening
def normalize_lengthening(text):
    if not isinstance(text, str):
        return text
    
    # Mengubah huruf berulang lebih dari 2 menjadi maksimal 2
    # contoh: bagussss → baguss, jelekkkk → jelekk
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    return text

In [ ]:
# 1.2 Menerapkan word lengthening pada kolom teks
df['teks_lengthening'] = df['teks'].apply(normalize_lengthening)

df[['teks', 'teks_lengthening']].head()

,teks,teks_lengthening
0,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh utk yg punya kebijakan publik negara ...
1,Tertibkan Media Online DPR Pemerintah Jangan S...,Tertibkan Media Online DPR Pemerintah Jangan S...
2,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lg kebijakan bebas visa truta...
3,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang
4,Kebebasan bersuara berpendapat memang dijamin ...,Kebebasan bersuara berpendapat memang dijamin ...


In [ ]:
# 1.3 Cek perubahan data
jumlah_berubah = (df['teks'] != df['teks_lengthening']).sum()

print("Jumlah data yang mengalami perubahan:", jumlah_berubah)

Jumlah data yang mengalami perubahan: 849


In [18]:
# 1.4 Simpan hasil setelah word lengthening
output_path = '../datapreprocessing-no-stem'
os.makedirs(output_path, exist_ok=True)

df.to_csv(
    os.path.join(output_path, 'data_after_word_lengthening_no_stem.csv'),
    index=False
)

print("data_after_word_lengthening_no_stem.csv berhasil disimpan.")

data_after_word_lengthening_no_stem.csv berhasil disimpan.


## Slang Normalization

In [21]:
# 2.1 Konfigurasi path
import os

# Naik 3 level ke folder PROJECT utama untuk memuat kamus
BASE_PROJECT = '../..'
KAMUS_PATH = os.path.join(BASE_PROJECT, 'kamus', 'colloquial_lexicon.csv')

# Input dari hasil word lengthening (eksperimen no_stem)
INPUT_PATH = '../datapreprocessing-no-stem/data_after_word_lengthening_no_stem.csv'

# Output directory
OUTPUT_DIR = '../datapreprocessing-no-stem'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [22]:
# 2.2 Load Kamus
import pandas as pd

df_kamus = pd.read_csv(KAMUS_PATH)
slang_dict = dict(zip(df_kamus['slang'].astype(str), df_kamus['formal'].astype(str)))

colloquial_to_formal = {
    # === NEGASI ===
    'enggak': 'tidak', 'nggak': 'tidak', 'gak': 'tidak', 'ga': 'tidak',
    'gk': 'tidak', 'tdk': 'tidak', 'bkn': 'bukan', 'bukannya': 'bukan',

    # === INTENSIFIER ===
    # PERUBAHAN: bgt/bgtd diubah ke 'sangat' (lebih natural saat ditranslasi ke 'very')
    # DIHAPUS: parah, gila, benar, betul — kata-kata ini punya makna sentimen sendiri
    #          dan harus dibiarkan ditranslasi apa adanya oleh Google Translate
    #          parah -> 'severe/serious', gila -> 'crazy/outrageous',
    #          benar -> 'true/correct', betul -> 'correct'
    'bgt': 'sangat', 'bgtd': 'sangat', 'amat': 'sangat',

    # === CONJUNCTION ===
    'tapi': 'tetapi', 'tp': 'tetapi', 'namun': 'tetapi',
    'kalo': 'jika', 'kalau': 'jika', 'kl': 'jika', 'klu': 'jika',

    # === COMMON SLANG ===
    'aja': 'saja', 'udah': 'sudah', 'udh': 'sudah', 'dah': 'sudah',
    'sdh': 'sudah', 'blm': 'belum', 'lg': 'lagi', 'yg': 'yang',
    'y': 'yang', 'dgn': 'dengan', 'dg': 'dengan', 'krn': 'karena',
    'bs': 'bisa', 'kyk': 'seperti', 'kayak': 'seperti',
    'gini': 'begini', 'gitu': 'begitu', 'nih': 'ini', 'tu': 'itu',
    'itu': 'itu', 'ini': 'ini', 'sih': 'sih', 'lah': 'lah',
    'kah': 'kah', 'dong': 'dong', 'deh': 'deh',

    # === PRONOUNS ===
    'aq': 'aku', 'gw': 'saya', 'gue': 'saya', 'gua': 'saya',
    'elu': 'kamu', 'lu': 'kamu', 'km': 'kamu',

    # === TIME/PLACE ===
    'skrg': 'sekarang', 'ntar': 'nanti', 'kemaren': 'kemarin',
}

def two_step_normalize(word, slang_dict, formal_dict):
    word_lower = word.lower()
    colloquial = slang_dict.get(word_lower, word_lower)
    formal = formal_dict.get(colloquial, colloquial)
    return formal

print(f"Total slang entries: {len(slang_dict)}")
print(f"Total colloquial→formal mappings: {len(colloquial_to_formal)}")

Total slang entries: 4331
Total colloquial→formal mappings: 54


In [23]:
# 2.3 Load Data Hasil Word Lengthening
df = pd.read_csv(INPUT_PATH)
print(f"Dataset berhasil dimuat: {len(df)} baris.")

Dataset berhasil dimuat: 13192 baris.


In [24]:
# 2.4 Fungsi Normalisasi Slang (Two-Step + Aturan Kapitalisasi VADER)
def normalize_slang(text, slang_dict, formal_dict):
    if not isinstance(text, str):
        return text
    
    words = text.split()
    normalized_words = []
    
    for word in words:
        # Two-step normalization
        normalized = two_step_normalize(word, slang_dict, formal_dict)
        
        # Aturan Kapitalisasi khusus VADER
        if word.isupper() and len(word) > 1:
            # FULL KAPITAL -> pertahankan agar VADER mendeteksi penekanan/intensitas
            normalized = normalized.upper()
        else:
            # Hanya 1-2 kapital, Title Case, atau lowercase -> jadikan semua lowercase
            normalized = normalized.lower()
        
        normalized_words.append(normalized)
    
    return ' '.join(normalized_words)

In [25]:
# 2.5 Menerapkan Normalisasi
df['teks_normalized'] = df['teks_lengthening'].apply(
    lambda x: normalize_slang(x, slang_dict, colloquial_to_formal)
)

print("\nSampel Perubahan (Two-Step):")
print(df[['teks_lengthening', 'teks_normalized']].head(10))


Sampel Perubahan (Two-Step):
                                    teks_lengthening  \
0  ADIL loh utk yg punya kebijakan publik negara ...   
1  Tertibkan Media Online DPR Pemerintah Jangan S...   
2  harus dievaluasi lg kebijakan bebas visa truta...   
3     jangan ngambang aturan logis apa undang undang   
4  Kebebasan bersuara berpendapat memang dijamin ...   
5  Anggota Komisi IX DPR Okky Asokawati Evaluasi ...   
6  Anggota DPR RI Komisi II Dari Fraksi NasDem Ge...   
7  Wapres Minta Kebijakan Bebas Visa Dievaluasi c...   
8  Asal Sesuai Undang Undang Anggota DPR Ini Setu...   
9  Saat memberikan makanan bergizi bantuan Kemenk...   

                                     teks_normalized  
0  ADIL loh untuk yang punya kebijakan publik neg...  
1  tertibkan media online DPR pemerintah jangan s...  
2  harus dievaluasi lagi kebijakan bebas visa ter...  
3     jangan ngambang aturan logis apa undang undang  
4  kebebasan bersuara berpendapat memang dijamin ...  
5  anggota komisi IX DP

In [26]:
# 2.6 Simpan hasil setelah slang normalization
df.to_csv(
    os.path.join(OUTPUT_DIR, 'data_after_slang_normalization_no_stem.csv'),
    index=False
)
print("\n data_after_slang_normalization_no_stem.csv berhasil disimpan.")


 data_after_slang_normalization_no_stem.csv berhasil disimpan.


In [27]:
# 2.7 Test fungsi preservasi kapitalisasi + two-step mapping
test_cases = [
    "utk apa YG bgtt",
    "UTK APA YG BGTT",
    "Utk Apa Yg Bgtt",
    "DPR harus evaluasi LG kebijakan ini",
    "ga enak tapi bgt sih",
    "GAK SUKA TAPI BAGUS"
]

print("Test Two-Step Normalization + Preservasi Kapitalisasi:\n")
for test in test_cases:
    result = normalize_slang(test, slang_dict, colloquial_to_formal)
    print(f"Original:   {test}")
    print(f"Normalized: {result}")
    print("-" * 60)

Test Two-Step Normalization + Preservasi Kapitalisasi:

Original:   utk apa YG bgtt
Normalized: untuk apa YANG banget
------------------------------------------------------------
Original:   UTK APA YG BGTT
Normalized: UNTUK APA YANG BANGET
------------------------------------------------------------
Original:   Utk Apa Yg Bgtt
Normalized: untuk apa yang banget
------------------------------------------------------------
Original:   DPR harus evaluasi LG kebijakan ini
Normalized: DPR harus evaluasi LAGI kebijakan ini
------------------------------------------------------------
Original:   ga enak tapi bgt sih
Normalized: tidak enak tetapi banget sih
------------------------------------------------------------
Original:   GAK SUKA TAPI BAGUS
Normalized: TIDAK SUKA TETAPI BAGUS
------------------------------------------------------------


## Tokenization

In [28]:
# 3.1 Load data hasil slang normalization
import pandas as pd

df = pd.read_csv('../datapreprocessing-no-stem/data_after_slang_normalization_no_stem.csv')

print("Dataset berhasil dimuat.")
print("Jumlah data:", len(df))
df.head()

Dataset berhasil dimuat.
Jumlah data: 13192


,no,timestamp,teks,teks_lengthening,teks_normalized
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,Tertibkan Media Online DPR Pemerintah Jangan S...,tertibkan media online DPR pemerintah jangan s...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lagi kebijakan bebas visa ter...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,Kebebasan bersuara berpendapat memang dijamin ...,kebebasan bersuara berpendapat memang dijamin ...


In [29]:
# 3.2 Fungsi Tokenization
import re

def tokenize_text(text):
    if not isinstance(text, str):
        return []
    
    # Pisahkan kata dan tanda baca
    tokens = re.findall(r'\w+|[^\w\s]', text, flags=re.UNICODE)
    
    # Hapus token kosong
    tokens = [t for t in tokens if t.strip()]
    
    return tokens

In [30]:
# 3.3 Menerapkan tokenization
df['tokens'] = df['teks_normalized'].apply(tokenize_text)

df[['teks_normalized', 'tokens']].head()

,teks_normalized,tokens
0,ADIL loh untuk yang punya kebijakan publik neg...,"[ADIL, loh, untuk, yang, punya, kebijakan, pub..."
1,tertibkan media online DPR pemerintah jangan s...,"[tertibkan, media, online, DPR, pemerintah, ja..."
2,harus dievaluasi lagi kebijakan bebas visa ter...,"[harus, dievaluasi, lagi, kebijakan, bebas, vi..."
3,jangan ngambang aturan logis apa undang undang,"[jangan, ngambang, aturan, logis, apa, undang,..."
4,kebebasan bersuara berpendapat memang dijamin ...,"[kebebasan, bersuara, berpendapat, memang, dij..."


In [31]:
# 3.4 Cek hasil tokenization
print("Contoh token:")
for i in range(5):
    print(df['tokens'].iloc[i])

Contoh token:
['ADIL', 'loh', 'untuk', 'yang', 'punya', 'kebijakan', 'publik', 'negara', 'ingat', 'yang', 'ini', '!', '!']
['tertibkan', 'media', 'online', 'DPR', 'pemerintah', 'jangan', 'sporadis', 'apalagi', 'selektif', 'hanya', 'kepada', 'media', 'yang', 'berseberangan']
['harus', 'dievaluasi', 'lagi', 'kebijakan', 'bebas', 'visa', 'terutama', 'untuk', 'negara', 'tiongkok', 'pak', '!', '!', 'bahaya', 'tersembunyi', 'martabat', 'RI']
['jangan', 'ngambang', 'aturan', 'logis', 'apa', 'undang', 'undang']
['kebebasan', 'bersuara', 'berpendapat', 'memang', 'dijamin', 'UU', 'tetapi', 'kebebasan', 'tersebut', 'tidak', 'harus', 'kebablasan', 'sehingga', 'menabrak', 'aturan', 'aturan', 'logis']


In [34]:
# 3.5 Simpan hasil tokenization
import os

output_path = '../datapreprocessing-no-stem'
os.makedirs(output_path, exist_ok=True)

df.to_csv(
    os.path.join(output_path, 'data_after_tokenization_no_stem.csv'),
    index=False
)

print("data_after_tokenization_no_stem.csv berhasil disimpan.")

data_after_tokenization_no_stem.csv berhasil disimpan.


# FORMATING DATA

In [35]:
# 4.1 Load data hasil tokenization
import pandas as pd
import os
import ast

df = pd.read_csv('../datapreprocessing-no-stem/data_after_tokenization_no_stem.csv')

print("Dataset berhasil dimuat.")
print("Jumlah data:", len(df))
print("Kolom:", df.columns.tolist())
df.head()

Dataset berhasil dimuat.
Jumlah data: 13192
Kolom: ['no', 'timestamp', 'teks', 'teks_lengthening', 'teks_normalized', 'tokens']


,no,timestamp,teks,teks_lengthening,teks_normalized,tokens
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...,"['ADIL', 'loh', 'untuk', 'yang', 'punya', 'keb..."
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,Tertibkan Media Online DPR Pemerintah Jangan S...,tertibkan media online DPR pemerintah jangan s...,"['tertibkan', 'media', 'online', 'DPR', 'pemer..."
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lagi kebijakan bebas visa ter...,"['harus', 'dievaluasi', 'lagi', 'kebijakan', '..."
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang,"['jangan', 'ngambang', 'aturan', 'logis', 'apa..."
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,Kebebasan bersuara berpendapat memang dijamin ...,kebebasan bersuara berpendapat memang dijamin ...,"['kebebasan', 'bersuara', 'berpendapat', 'mema..."


In [36]:
# 4.2 Pilih Kolom yang Dibutuhkan
df_final = df[['no', 'timestamp', 'teks', 'tokens']].copy() 

In [37]:
# 4.3 Convert List Token ke Teks (String)
def safe_join(x):
    # Jika datanya sudah berupa list 
    if isinstance(x, list):
        return ' '.join(x)
    
    # Jika datanya berupa string list "['a', 'b']"
    try:
        x_list = ast.literal_eval(x)
        if isinstance(x_list, list):
            return ' '.join(x_list)
    except:
        pass
    
    # Jika gagal, kembalikan aslinya
    return x

# Terapkan fungsi join pada kolom tokens
df_final['teks_processed'] = df_final['tokens'].apply(safe_join) 


In [38]:
# 4.4 Hapus Kolom List Lama (biar tidak double)
df_final = df_final.drop(columns=['tokens']) 

In [39]:
# 4.5 Validasi
print("Validasi Data Final Preprocessing")
print(f"Jumlah data: {len(df_final)}")
print(f"Kolom final: {df_final.columns.tolist()}")
print(f"Data kosong: {df_final['teks_processed'].isna().sum()}")

# Cek sampel
print("Sampel Data")
print(df_final[['teks', 'teks_processed']].head(3))

Validasi Data Final Preprocessing
Jumlah data: 13192
Kolom final: ['no', 'timestamp', 'teks', 'teks_processed']
Data kosong: 0
Sampel Data
                                                teks  \
0  ADIL loh utk yg punya kebijakan publik negara ...   
1  Tertibkan Media Online DPR Pemerintah Jangan S...   
2  harus dievaluasi lg kebijakan bebas visa truta...   

                                      teks_processed  
0  ADIL loh untuk yang punya kebijakan publik neg...  
1  tertibkan media online DPR pemerintah jangan s...  
2  harus dievaluasi lagi kebijakan bebas visa ter...  


In [40]:
# 4.6 Simpan sebagai Dataset Final 
output_dir = '../datapreprocessing-no-stem'
os.makedirs(output_dir, exist_ok=True)

df_final.to_csv(
    os.path.join(output_dir, 'data_preprocessing_final_no_stem.csv'),
    index=False,
    encoding='utf-8'
)

print("data_preprocessing_final_no_stem.csv berhasil disimpan!")
print(f"Lokasi: {output_dir}/data_preprocessing_final_no_stem.csv")

data_preprocessing_final_no_stem.csv berhasil disimpan!
Lokasi: ../datapreprocessing-no-stem/data_preprocessing_final_no_stem.csv
